# Construir un ejemplo de aprendizaje para procesamiento de minerales

**Pregunta:** ¿Qué etiquetas físicas y errores de interpolación produce un Ridge pequeño entrenado con el simulador?

Nivel: 1. Requisitos previos: Ninguno; comienza aquí. Tiempo de lectura: 20 minutos.



## Objetivos

Construir procedencia física de variables y objetivos y ajustar Ridge con partición 48/16 de 64 estados. Tres pruebas de casos establecen significado de etiquetas antes de examinar un modelo y una pregunta.

## Preparación

```bash
pip install numpy==2.2.6 scipy==1.17.1 pandas==2.2.3 matplotlib==3.11.2 nbclient==0.11.0 nbformat==5.11.1 ipykernel==7.4.0 SALib==1.6.0 scikit-learn==1.5.2 ruff==0.15.18 pytest==9.1.1
```

El pipeline OreFlow se importa desde `data-pipeline/`; este notebook nunca instala paquetes.

Importar las herramientas fijadas y resolver las entradas relativas a la carpeta del notebook.

In [ ]:
# Importar herramientas de análisis y registrar sus versiones exactas.
import importlib.metadata
import json
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

repo = Path("../../../").resolve()
sys.path.insert(0, str(repo / "data-pipeline"))
for package in ("numpy", "scipy", "pandas", "matplotlib", "nbclient", "nbformat", "ipykernel"):
    print(package, importlib.metadata.version(package))
legend_style = dict(loc="lower center", bbox_to_anchor=(0.5, 1.02), frameon=False)


## Datos

Responsable: autores de OreFlow, licencia MIT. Se lee `data/derived/contract/operating_contract.json` y el catálogo aporta cuatro definiciones de casos. `build_design` genera 64 etiquetas mediante el motor de circuito. No se leen puntuaciones de learning.json ni modelos desplegados. Estados y factores minerales son entradas sintéticas del modelo, no mediciones inventadas. Fuente: https://github.com/fsantibanezleal/CAOS_OreFlow.

## Método

La primera regresión ajusta solo recuperación. Cuatro casos aportan dieciséis estados cada uno; las variables describen mineral y operación sin identificador de caso explícito. Ridge aprende con 48 filas después de ajustar StandardScaler solo en ellas. Dieciséis filas reservadas aportan RMSE, MAE y R2. Tres resoluciones nominales establecen procedencia de etiquetas. Las propiedades pueden codificar identidad indirectamente; esta prueba es interpolación entre casos representados, no otra población mineral.

Crear un primer conjunto supervisado con sesenta y cuatro resoluciones acotadas. Cuatro circuitos aportan dieciséis estados cada uno. Las variables describen mineral y operación sin incluir identificador de caso. Los objetivos son recuperación, enriquecimiento logarítmico y energía total. Aquí se ajusta solo recuperación, una tarea comprensible que muestra de dónde provienen las otras etiquetas. El generador respeta contrato y usa semilla fija para repetir el diseño.

In [ ]:
# Construir el diseño de aprendizaje desde las salidas físicas del circuito.
from pipeline.cases.catalog import CASE_BY_ID
from pipeline.engine.circuit import simulate
from pipeline.methods.learning import build_design, interpolation_split, leave_one_case_out
from pipeline.methods.learning import leave_one_group_out, transfer_groups, FEATURES, TARGETS
import contextlib
import io

contract = json.loads(
    (repo / "data/derived/contract/operating_contract.json").read_text(encoding="utf-8")
)
case_ids = ("copper_porphyry_soft", "copper_porphyry_hard", "copper_oxide", "iron_magnetite_fine")
selected = tuple((CASE_BY_ID[name] for name in case_ids))
with contextlib.redirect_stdout(io.StringIO()):
    design = build_design(contract, cases=selected, per_case=16, seed=20261031)
x, y, case_of = (design["x"], design["y"], design["case"])
print("Filas del diseño:", len(x))
print("Variables:", len(FEATURES))
print("Objetivos:", TARGETS)


Inspeccionar directamente una etiqueta nominal del pórfido blando. Recuperación es fracción de material valioso recuperado; energía específica se normaliza por masa tratada. El aprendiz recibe una salida solo si la eliges como objetivo: aquí recuperación. Resolver por separado demuestra procedencia física antes de entregar matrices al estimador. También establece que entrenamiento no oculta una respuesta medida de planta.

In [ ]:
# Inspeccionar la etiqueta nominal de `copper_porphyry_soft` antes de agrupar sus filas.
case = CASE_BY_ID["copper_porphyry_soft"]
point = case.nominal
reference = simulate(case.ore, case.plant, point)
print("Caso de referencia:", case.id)
print("Recuperación de referencia (%):", round(reference.metrics["recovery_pct"], 3))
print("Energía de referencia (kWh/t):", round(reference.metrics["specific_energy_total_kwh_t"], 3))


Resolver cobre oxidado con su propia definición nominal. Sus propiedades difieren del pórfido; configuraciones operativas cercanas no implican etiquetas cercanas entre casos. Aunque falta el identificador, las propiedades físicas pueden codificar señales fuertes de identidad. Por eso una buena prueba aleatoria por filas no equivale automáticamente a generalizar a un mineral desconocido.

In [ ]:
# Inspeccionar la etiqueta nominal de `copper_oxide` antes de agrupar sus filas.
case = CASE_BY_ID["copper_oxide"]
point = case.nominal
reference = simulate(case.ore, case.plant, point)
print("Caso de referencia:", case.id)
print("Recuperación de referencia (%):", round(reference.metrics["recovery_pct"], 3))
print("Energía de referencia (kWh/t):", round(reference.metrics["specific_energy_total_kwh_t"], 3))


Calcular magnetita con su diagrama de separación, sin imponer un modelo exclusivo de flotación. El vector incluye indicadores de etapas instaladas. Comparar la etiqueta con los ejemplos de cobre enseña el alcance de una representación común. Unidades de recuperación compartidas permiten un objetivo de regresión, pero no vuelven idénticos los mecanismos minerales y de separación.

In [ ]:
# Inspeccionar la etiqueta nominal de `iron_magnetite_fine` antes de agrupar sus filas.
case = CASE_BY_ID["iron_magnetite_fine"]
point = case.nominal
reference = simulate(case.ore, case.plant, point)
print("Caso de referencia:", case.id)
print("Recuperación de referencia (%):", round(reference.metrics["recovery_pct"], 3))
print("Energía de referencia (kWh/t):", round(reference.metrics["specific_energy_total_kwh_t"], 3))


Reserva un cuarto de las filas de cada caso mediante el particionador de interpolación. Entrenamiento y prueba contienen los cuatro casos: se pregunta por estados nuevos de circuitos ya representados. El inventario impreso revela el límite antes de puntuar. Escala dentro del entrenamiento; usar medias del diseño completo introduciría información de prueba en la representación aun sin filtrar una etiqueta objetivo.

In [ ]:
# Construir tres preguntas de validación sobre exactamente el mismo diseño.
train, test = interpolation_split(case_of, test_fraction=0.25, seed=20261031)
case_fold = next((f for f in leave_one_case_out(case_of) if f[0] == "copper_porphyry_soft"))
group_fold = leave_one_group_out(case_of, transfer_groups(selected))[0]
splits = {
    "Interpolación": (train, test),
    "Caso reservado": case_fold[1:],
    "Grupo mineral reservado": group_fold[1:],
}
for protocol, (training, testing) in splits.items():
    print(protocol, len(training), len(testing))
    print("Casos de entrenamiento:", sorted(set(case_of[training].tolist())))
    print("Casos de prueba:", sorted(set(case_of[testing].tolist())))
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

scores, predictions = ([], {})
splits = {list(splits)[0]: next(iter(splits.values()))}


Ajustar Ridge después de estandarizar solo con entrenamiento y predice las etiquetas reservadas. La penalización estabiliza un ajuste lineal con variables correlacionadas. Conservar alpha fijo, sin buscarlo con puntuaciones de prueba. Calcular RMSE, MAE y R² desde predicciones. Un R² negativo indica peor desempeño que la referencia de media de prueba y es informativo; no justifica ocultar el pliegue.

In [ ]:
# Ridge penaliza los coeficientes al cuadrado después de escalar.
for protocol, (training, testing) in splits.items():
    model = make_pipeline(StandardScaler(), Ridge(alpha=1.0))
    model.fit(x[training], y[training, 0])
    predicted = model.predict(x[testing])
    print("Rango de predicción (%):", np.round([predicted.min(), predicted.max()], 3))
    print("Rango de referencia (%):", np.round([y[testing, 0].min(), y[testing, 0].max()], 3))
    predictions["Ridge", protocol] = (testing, predicted)
    scores.append(
        {
            "Modelo": "Ridge",
            "Protocolo": protocol,
            "RMSE (pp)": float(np.sqrt(mean_squared_error(y[testing, 0], predicted))),
            "MAE (pp)": float(mean_absolute_error(y[testing, 0], predicted)),
            "R2": float(r2_score(y[testing, 0], predicted)),
        }
    )
display(pd.DataFrame(scores).tail(3).round(3))


Dibujar el error de interpolación de este único estimador. La lección inicial contiene un protocolo y un modelo ajustado; la barra resume una predicción concreta, no una clasificación bibliográfica. Usar puntos porcentuales porque mide diferencias de recuperación, no cambio porcentual relativo. Leer junto a rangos de predicción y referencia para juzgar la magnitud.

In [ ]:
# Comparar el RMSE de recuperación dentro de cada pregunta de validación.
plt.rcParams.update({"font.size": 18, "axes.titlepad": 40})
score = pd.DataFrame(scores)
fig, ax = plt.subplots(figsize=(6.4, 3.8))
score.pivot(index="Modelo", columns="Protocolo", values="RMSE (pp)").plot.barh(ax=ax)
ax.set(xlabel="RMSE de\nrecuperación\n(pp)", ylabel="", title="Error con diseño común")
ax.tick_params(axis="x", rotation=0)
handles, legend_names = ax.get_legend_handles_labels()
legend_short = {"Caso reservado": "Caso", "Grupo mineral reservado": "Grupo mineral"}
ax.legend(
    handles,
    [legend_short.get(name, name) for name in legend_names],
    fontsize=18,
    ncol=2,
    **legend_style,
)
fig.subplots_adjust(left=0.4, right=0.96, bottom=0.23, top=0.72)
print("Filas RMSE calculadas:", len(score))


La barra es descriptiva y no tiene intervalo de confianza: se usa una partición determinista. Las sesenta y cuatro etiquetas comparten cuatro definiciones y no son observaciones geológicas independientes. El modelo aproxima el motor. Antes de agregar estimadores no lineales, confirma unidades, representación de casos en ambas particiones y que el estandarizador haya visto solamente filas de entrenamiento.

## Resultados

**Respuesta:** El diseño común contiene <!-- result:Filas del diseño -->0<!-- /result --> estados simulados. Ridge se ajusta con <!-- result:Filas de entrenamiento -->0<!-- /result --> filas y predice <!-- result:Filas de prueba interpolada -->0<!-- /result --> filas reservadas, con RMSE <!-- result:RMSE interpolado de Ridge (pp) -->0<!-- /result --> pp. Cada caso sigue representado en entrenamiento; se establece interpolación de etiquetas del motor, no predicción de mineral ausente ni planta medida.

Resumir cantidades reales de diseño, entrenamiento y prueba interpolada junto al RMSE de recuperación de Ridge.

In [ ]:
# Resumir las salidas calculadas que sostienen la interpretación de esta lección.
score_by_key = {(row["Modelo"], row["Protocolo"]): row for row in scores}
answer_values = {
    "Filas del diseño": len(x),
    "RMSE interpolado de Ridge (pp)": score_by_key["Ridge", "Interpolación"]["RMSE (pp)"],
}
answer_values.update(
    {"Filas de entrenamiento": len(train), "Filas de prueba interpolada": len(test)}
)
answer_values = {label: round(float(number), 3) for label, number in answer_values.items()}
print(json.dumps(answer_values, ensure_ascii=False))


## Resumen

El ajuste inicial usa 48 filas para predecir dieciséis reservadas dentro de 64 estados. Procedencia y unidades preceden cualquier orden de modelos. En esta pregunta de interpolación no falta un caso mineral al entrenar.

## Referencias

- https://doi.org/10.1023/A:1010933404324
- https://doi.org/10.1023/A:1010933404324